Final implementation of Static Baseline For Final Comparison of Results.  This static Baseline was chosen because as shown in the class imbalance Nontarget Data is most of the Dataset so it should already prove to be accurate. We will see how it compares to the LDA and P300NN. 

In [4]:
import mne
import numpy as np
from numpy import matlib as mb
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import warnings
from sklearn import metrics
from collections import defaultdict
import regex as re
from sklearn.metrics import accuracy_score
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
from sklearn.metrics import roc_auc_score
from models.Baseline import BaselineClassifier
from src.Dataset import Dataset
from src.Speller import Speller
warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("WARNING")

Divide and Separate Training and Testing Paths

In [2]:
def group_paths_by_participant(data_paths):
    participant_files = defaultdict(lambda: {'train': [], 'test': []})
    for path in data_paths:
        parts = path.split(os.sep)
        participant_id = parts[6]

        if 'Train' in path or 'train' in path:
            participant_files[participant_id]['train'].append(path)
        elif 'Test' in path or 'test' in path:
            participant_files[participant_id]['test'].append(path)
        else:
            print(f"Unclassified path: {path}")

    return participant_files


In [3]:
important_channels = ['EEG_Fz', 'EEG_Cz', 'EEG_Pz', 'EEG_P3',
                      'EEG_PO7', 'EEG_PO8', 'EEG_P4', 'EEG_Oz']

#first using default values
ds = Dataset(
    glob_path=os.path.join(project_root, "data", "*", "*", "*", "CB", "*"),
    tmin=0,
    tmax=0.8, 
)

participant_files = group_paths_by_participant(ds.data_paths)
participants = list(participant_files.keys())

Training LDA with Preprocessing

In [5]:
studyL_results = []

for participant_id in participant_files.keys():

    print(f"\nProcessing participant {participant_id}")

    train_files = participant_files[participant_id]['train']
    test_files  = participant_files[participant_id]['test']

    X_train, y_train = [], []
    X_test, y_test   = [], []

    # Load train data
    for path in train_files:
        ds = Dataset(
            path,
            important_channels=important_channels,
            sample_rate=256,
            tmin=0,
            tmax=0.8,
            use_car=True,
            notch_filter=True
        )
        X, y = ds[0]
        X_train.append(X)
        y_train.append(y)

    # Load test data
    for path in test_files:
        ds = Dataset(
            path,
            important_channels=important_channels,
            sample_rate=256,
            tmin=0,
            tmax=0.8,
            use_car=True,
            notch_filter=True
        )
        X, y = ds[0]
        X_test.append(X)
        y_test.append(y)

    X_train = np.vstack(X_train)
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    y_train = np.concatenate(y_train)

    X_test = np.vstack(X_test)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    y_test = np.concatenate(y_test)

    # Use baseline
    clf = BaselineClassifier()
    scores = clf.predict_scores(X_test_flat, y_test)
    y_pred = np.zeros(len(y_test), dtype=int)
    acc = accuracy_score(y_test, y_pred)
    auc = 0.5 # always 0.5

    print(f"Participant {participant_id} Baseline Accuracy: {acc:.3f}, AUC: {auc:.3f}")

    character_accuracies = []
    for path in test_files:
        ds = Dataset(
            path,
            important_channels=important_channels,
            sample_rate=256,
            tmin=0,
            tmax=0.8,
            notch_filter=True,
            use_car=True
        )
        X_test_specific, y_test_specific = ds[0]
        X_test_specific_flat = X_test_specific.reshape(X_test_specific.shape[0], -1)

        scores_specific = np.zeros(X_test_specific_flat.shape[0])

        raw = ds.raw
        epochs = ds.epochs
        char_ch_names = [ch for ch in raw.ch_names if re.match(r'^[A-Za-z0-9]+_\d+_\d+$', ch)]
        char_ch_indices = [raw.ch_names.index(ch) for ch in char_ch_names]

        curr_target_idx = raw.ch_names.index('CurrentTarget')
        phase_idx = raw.ch_names.index('PhaseInSequence')
        data = raw.get_data()
        stim_indices = np.where(data[phase_idx] == 2)[0]
        changes = np.diff(data[curr_target_idx], prepend=data[curr_target_idx][0]-1)
        target_onsets = stim_indices[np.isin(stim_indices, np.where(changes != 0)[0])]
        target_codes = data[curr_target_idx][target_onsets].astype(int)
        current_target_events = np.array([[onset, 0, code] for onset, code in zip(target_onsets, target_codes)])

        pcr = {
            'epochs': epochs,
            'raw_data': raw,
            'character_channels': char_ch_indices,
            'current_target_events': current_target_events
        }

        speller_grid = [
            "A","B","C","D","E","F","G","H","I","J","K","L","M",
            "N","O","P","Q","R","S","T","U","V","W","X","Y","Z",
            "_","1","2","3","4","5","6","7","8","9"
        ]
        speller = Speller(speller_grid)

        predictions = speller.run(pcr, clf=None, X=scores_specific, y=1)
        metrics = speller.get_metrics()
        character_accuracies.append(metrics['accuracy'])

    avg_char_acc = np.mean(character_accuracies)
    print(f"Average character accuracy: {avg_char_acc:.3f}")

    # Store results
    studyL_results.append({
        "participant": participant_id,
        "baseline_accuracy": acc,
        "baseline_auc": auc,
        "characcuracy": avg_char_acc,
    })

# Summary DataFrame
results_df = pd.DataFrame(studyL_results)
print("\nStudy L Baseline Summary")
print(results_df)
print("Average AUC:", results_df["baseline_auc"].mean())
print("Average Model Accuracy:", results_df["baseline_accuracy"].mean())
print("Average Character Accuracy Across Study L:", results_df["characcuracy"].mean())


Processing participant L_01
Accuracy: 0.8889
AUC: 0.5000
Participant L_01 Baseline Accuracy: 0.889, AUC: 0.500
Average character accuracy: 0.067

Processing participant L_02
Accuracy: 0.8889
AUC: 0.5000
Participant L_02 Baseline Accuracy: 0.889, AUC: 0.500
Average character accuracy: 0.067

Processing participant L_03
Accuracy: 0.8889
AUC: 0.5000
Participant L_03 Baseline Accuracy: 0.889, AUC: 0.500
Average character accuracy: 0.067

Processing participant L_04
Accuracy: 0.8889
AUC: 0.5000
Participant L_04 Baseline Accuracy: 0.889, AUC: 0.500
Average character accuracy: 0.000

Processing participant L_05
Accuracy: 0.8889
AUC: 0.5000
Participant L_05 Baseline Accuracy: 0.889, AUC: 0.500
Average character accuracy: 0.033

Processing participant L_06
Accuracy: 0.8889
AUC: 0.5000
Participant L_06 Baseline Accuracy: 0.889, AUC: 0.500
Average character accuracy: 0.067

Processing participant L_07
Accuracy: 0.8889
AUC: 0.5000
Participant L_07 Baseline Accuracy: 0.889, AUC: 0.500
Average char